# Project Assignment: Space Travel — Main Audio Pipeline

This notebook assembles the complete **40–60 s stereo WAV** (`final_soundtrack.wav`) by importing each group member's module:
- `synth.py` — Clean synthesizer transmission & confirmation chirp
- `tune.py` — Instrument detection & pitch deviation analysis
- `bandpass.py` — 6–8 kHz narrowband anomaly + IIR (bilinear/pre-warped) & FIR band-stop removal
- `broadband.py` — Broadband radiation storm noise + Welch/periodogram PSD estimation
- `Moon_entry.py` — Chebyshev Type I low-pass (bilinear/pre-warped), quantization reduction, and Hann-windowed dropouts
- `blackout.py` — Complete 2 s blackout + interpolation & SNR analysis
- `recovery.py` — Resampling, step-wise recovery, raised-cosine cross-fades, and stereo mastering

**Sampling Rate Justification (Nyquist):** `fs = 44100 Hz` gives a Nyquist frequency of `22050 Hz`, covering the full human audible range (~20 Hz–20 kHz) without aliasing. Any 48 kHz assets (`space_sound_placeholder.wav`, `piano.wav`) are anti-alias resampled to `44100 Hz` on load.

In [ ]:
import os
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from synth import generate_clean_transmission, generate_detection_beep
from tune import process_tuning_pipeline
from bandpass import apply_anomaly_phase, design_iir_bandstop, design_fir_bandstop
from broadband import apply_radiation_storm, compute_psd_welch, compute_psd_periodogram
from Moon_entry import moon_shadow, chebychev_filter
from blackout import apply_complete_blackout_phase, calculate_snr
from recovery import (
    load_audio,
    fit_duration,
    crossfade_concat,
    apply_gradual_recovery,
    master_and_save_stereo,
)

fs = 44100

# Optional: set NEWS_AUDIO_PATH to a recorded speech/news .wav file (20-25 s recommended).
# If not present, the synthesizer radio track is used as a fallback carrier.
NEWS_AUDIO_PATH = "news_feed.wav"
INSTRUMENT_PATH = "testing/instrument_sounds/guitar.wav"
AMBIENCE_PATH = "space_sound_placeholder.wav"

### Scene 1: Calm Opening Transmission (0–9 s)
Synthesized interstellar radio music lightly layered with resampled spaceship background ambience.

In [ ]:
full_music = generate_clean_transmission(fs=fs, use_full_song=True)
scene1_music = fit_duration(full_music[: int(9.0 * fs)], fs, 9.0)

if os.path.exists(AMBIENCE_PATH):
    _, ambience = load_audio(AMBIENCE_PATH, target_fs=fs)
    ambience_s1 = fit_duration(ambience, fs, 9.0)
    scene1 = scene1_music + 0.08 * ambience_s1
else:
    scene1 = scene1_music

print(f"Scene 1 duration: {len(scene1) / fs:.2f} s")
display(Audio(scene1, rate=fs))

### Scene 2: Instrument Sound & Detection Procedure (~9–13.6 s)
Plays the recorded instrument, classifies it with `tune.process_tuning_pipeline`, and emits a scanner confirmation beep.

In [ ]:
_, inst_audio = load_audio(INSTRUMENT_PATH, target_fs=fs)
inst_clip = fit_duration(inst_audio, fs, 4.0)

detected_instrument, pitch_dev_cents = process_tuning_pipeline(inst_clip, fs)
print(f"Detected instrument: {detected_instrument} | Pitch deviation from A4 (440 Hz): {pitch_dev_cents:.2f} cents")

beep = generate_detection_beep(fs=fs, duration=0.45)
scene2 = crossfade_concat([inst_clip, (beep, 0.03)], fs=fs)
print(f"Scene 2 duration: {len(scene2) / fs:.2f} s")
display(Audio(scene2, rate=fs))

### Scene 3: Gradual Anomaly Interference (6–8 kHz) & Band-Stop Filtering (~13.6–21.6 s)
6–8 kHz narrowband noise ramps in over the radio music, then a pre-warped IIR band-stop filter removes the anomaly.

In [ ]:
s3_start = int(9.0 * fs)
s3_clean = fit_duration(full_music[s3_start : s3_start + int(8.0 * fs)], fs, 8.0)

scene3, s3_noisy_only, anomaly_filters = apply_anomaly_phase(
    s3_clean,
    fs=fs,
    band=(6000.0, 8000.0),
    noise_level=0.22,
    order=4,
    fir_numtaps=401,
    cleanup_fraction=0.45,
    filter_type="iir",
)
print(f"Scene 3 duration: {len(scene3) / fs:.2f} s")
display(Audio(scene3, rate=fs))

### Scene 4: News Feed + Radiation Storm Near Chebychev-Moon (~21.6–29.6 s)
Loads `news_feed.wav` if present (or uses the radio synthesizer track) and overlays spectrally shaped broadband crackle/hiss.

In [ ]:
if os.path.exists(NEWS_AUDIO_PATH):
    _, news_base = load_audio(NEWS_AUDIO_PATH, target_fs=fs)
else:
    # Fallback carrier from later bars of the synthesized broadcast
    news_base = full_music[int(17.0 * fs) :]

news_27s = fit_duration(news_base, fs, 27.0)
s4_clean = news_27s[: int(8.0 * fs)]
s5_clean = news_27s[int(8.0 * fs) : int(16.0 * fs)]
s7_clean = news_27s[int(16.0 * fs) : int(27.0 * fs)]

scene4, storm_noise = apply_radiation_storm(s4_clean, fs=fs, snr_db=5.0, ramp_s=1.5)
print(f"Scene 4 duration: {len(scene4) / fs:.2f} s")
display(Audio(scene4, rate=fs))

### Scene 5: Moon Shadow Entry (~29.6–37.6 s)
Chebyshev Type I low-pass bandwidth cut + reduced quantization (8-bit) + progressive Hann-windowed dropouts.

In [ ]:
scene5, (b_cheb, a_cheb) = moon_shadow(s5_clean, fs=fs, dropouts=14, bits=8)
print(f"Scene 5 duration: {len(scene5) / fs:.2f} s")
display(Audio(scene5, rate=fs))

### Scene 6: Complete Blackout (~37.6–39.6 s) & Interpolation Analysis
Inserts 2.0 s of complete signal loss in the final audio track, while running `apply_complete_blackout_phase` to evaluate quantization SNR and spline interpolation.

In [ ]:
# Exact 2.0 s full signal loss for the movie audio track
scene6 = np.zeros(int(2.05 * fs), dtype=float)

# Technical evaluation of quantization + dropout interpolation from blackout.py
test_segment = s5_clean[: int(4.0 * fs)]
blackout_eval, reconstructed_eval = apply_complete_blackout_phase(test_segment, fs=fs, blackout_start_time=1.0)
snr_quant = calculate_snr(test_segment[: int(1.0 * fs)], blackout_eval[: int(1.0 * fs)])
print(f"4-bit quantization SNR (pre-blackout): {snr_quant:.2f} dB")

### Scene 7: Gradual Recovery & Return to Normal (~39.6–50.6 s)
Step-wise restoration from heavy bandwidth reduction (8 kHz effective fs, 4-bit) up to full 44.1 kHz, 16-bit clarity.

In [ ]:
scene7 = apply_gradual_recovery(s7_clean, fs=fs)
print(f"Scene 7 duration: {len(scene7) / fs:.2f} s")
display(Audio(scene7, rate=fs))

### Master Assembly & Export (`final_soundtrack.wav`)
Cross-fades all 7 scenes seamlessly (with short 25 ms fades at the 2 s blackout edges so the blackout stays truly silent), normalizes headroom, and exports stereo 16-bit WAV.

In [ ]:
segments = [
    scene1,
    (scene2, 0.15),
    (scene3, 0.20),
    (scene4, 0.20),
    (scene5, 0.20),
    (scene6, 0.025),
    (scene7, 0.025),
]

final_mono = crossfade_concat(segments, fs=fs, crossfade_s=0.15)
final_stereo = master_and_save_stereo(final_mono, "final_soundtrack.wav", fs=fs, target_peak=0.9)

total_dur = len(final_mono) / fs
print(f"Final soundtrack written to final_soundtrack.wav | Duration: {total_dur:.2f} s | Shape: {final_stereo.shape}")
display(Audio(final_mono, rate=fs))

### Technical Verification Plots (Table A.2 Mandatory Components)
1. **Full-track spectrogram** across all 7 scene phases
2. **IIR vs. FIR 6–8 kHz band-stop frequency response** (bilinear pre-warped Butterworth vs. Hamming-windowed FIR)
3. **z-plane pole-zero plot** for recursive filter stability ($|z| < 1$)
4. **Welch PSD vs. Periodogram** of the radiation storm noise (window length & $\Delta f$ trade-off)

In [ ]:
# 1. Full-track spectrogram
nperseg_spec = 2048  # df = 44100 / 2048 ~= 21.5 Hz, dt ~= 46 ms
f_spec, t_spec, Sxx = signal.spectrogram(final_mono, fs=fs, window="hann", nperseg=nperseg_spec, noverlap=1024)

plt.figure(figsize=(14, 5))
plt.pcolormesh(t_spec, f_spec, 10 * np.log10(Sxx + 1e-12), shading="gouraud", cmap="magma")
plt.colorbar(label="Power [dB]")
plt.ylim(0, 14000)
plt.xlabel("Time [s]")
plt.ylabel("Frequency [Hz]")
plt.title("Spectrogram of Final 40-60 s Soundtrack Across Scene Phases")

boundaries = [
    (0.0, "1. Opening"),
    (9.0, "2. Instrument"),
    (13.5, "3. 6-8kHz Anomaly"),
    (21.3, "4. Storm"),
    (29.1, "5. Moon Entry"),
    (36.9, "6. Blackout"),
    (39.0, "7. Recovery"),
]
for t_mark, label in boundaries:
    plt.axvline(t_mark, color="cyan", linestyle="--", alpha=0.8)
    plt.text(t_mark + 0.3, 13000, label, color="white", fontsize=8, rotation=90, va="top")
plt.tight_layout()
plt.show()

# 2. IIR vs FIR Band-stop Response + 3. z-plane Pole-Zero Plot + 4. PSD Comparison
b_iir, a_iir = anomaly_filters["iir_bandstop"]
b_fir, a_fir = anomaly_filters["fir_bandstop"]
w_iir, h_iir = signal.freqz(b_iir, a_iir, worN=2048, fs=fs)
w_fir, h_fir = signal.freqz(b_fir, a_fir, worN=2048, fs=fs)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(w_iir, 20 * np.log10(np.abs(h_iir) + 1e-12), label="IIR Butterworth (order 8)")
axes[0].plot(w_fir, 20 * np.log10(np.abs(h_fir) + 1e-12), label="FIR Hamming (401 taps)", alpha=0.85)
axes[0].axvspan(6000, 8000, color="red", alpha=0.1, label="6-8 kHz Stopband")
axes[0].set(title="IIR vs FIR Band-Stop Magnitude", xlabel="Frequency [Hz]", ylabel="Magnitude [dB]", xlim=(0, 15000), ylim=(-60, 5))
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Pole-zero plot for Chebyshev Type I filter from Moon_entry.py
z_c, p_c, _ = signal.tf2zpk(b_cheb, a_cheb)
theta = np.linspace(0, 2 * np.pi, 400)
axes[1].plot(np.cos(theta), np.sin(theta), "k--", alpha=0.5, label="Unit Circle")
axes[1].scatter(np.real(z_c), np.imag(z_c), marker="o", facecolors="none", edgecolors="b", label="Zeros")
axes[1].scatter(np.real(p_c), np.imag(p_c), marker="x", color="r", label="Poles")
axes[1].set(title=f"Chebyshev z-Plane (max |p|={np.max(np.abs(p_c)):.4f})", xlabel="Re(z)", ylabel="Im(z)", aspect="equal")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Welch PSD vs Periodogram for Radiation Storm noise
f_per, p_per = compute_psd_periodogram(storm_noise, fs=fs)
f_wel, p_wel = compute_psd_welch(storm_noise, fs=fs, nperseg=1024)
axes[2].plot(f_per, 10 * np.log10(p_per + 1e-15), alpha=0.35, label="Periodogram")
axes[2].plot(f_wel, 10 * np.log10(p_wel + 1e-15), linewidth=1.8, label="Welch (nperseg=1024)")
axes[2].set(title="Radiation Storm Noise PSD", xlabel="Frequency [Hz]", ylabel="PSD [dB/Hz]", xlim=(0, 16000))
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()